In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model



# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 12
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase2"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [4]:
from huggingface_hub import hf_hub_download

TRAIN_FILES = [
    "Phase_5_LSI/Train/train.json",
]

train_file_paths = {}

for filename in TRAIN_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    train_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/train.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/train.json


In [5]:
from huggingface_hub import hf_hub_download

VAL_FILES = [
    "Phase_5_LSI/Train/dev.json",
]

val_file_paths = {}

for filename in VAL_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    val_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/dev.json


In [6]:
from huggingface_hub import hf_hub_download

TEST_FILES = [
    "Phase_5_LSI/Test/test.json",
    
]

test_file_paths = {}

for filename in TEST_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    test_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Test/test.json


In [7]:
import json
import pandas as pd

with open(train_file_paths["Phase_5_LSI/Train/train.json"], "r") as f:
    train_data = json.load(f)

with open(val_file_paths["Phase_5_LSI/Train/dev.json"], "r") as f:
    val_data = json.load(f)
with open(test_file_paths["Phase_5_LSI/Test/test.json"], "r") as f:
    test_data = json.load(f)
    

In [8]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [9]:
import numpy as np
import pickle

# Paths
bpe_path = "/kaggle/input/datasets/sai123451/tokenizer/project_bpe_16k.pkl"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    tokenizer = pickle.load(f)
print("BPE vocab size:", len(tokenizer.itos))


BPE vocab size: 7999


In [10]:
converter = ToTokensConverter(tokenizer.itos)

In [11]:
import json
import torch
from torch.utils.data import Dataset, DataLoader

MAX_CHUNK_LEN = 1024
CHUNK_OVERLAP = 64
MAX_CHUNKS = 16

NUM_LABELS = 100  # 100 target IPC statutes in the paper


class LSIDataset(Dataset):
    def __init__(
        self,
        json_path,
        converter=converter,
        max_chunk_len=MAX_CHUNK_LEN,
        overlap=CHUNK_OVERLAP,
        max_chunks=MAX_CHUNKS,
        has_labels=True,
        num_labels=NUM_LABELS,
    ):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.converter = converter
        self.max_chunk_len = max_chunk_len
        self.overlap = overlap
        self.max_chunks = max_chunks
        self.has_labels = has_labels
        self.num_labels = num_labels

        assert 0 <= overlap < max_chunk_len - 2
        assert max_chunk_len >= 3

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        token_ids = self.converter.to_tokens([item["text"]])[0]

        pad_id = 0
        bos_id = 2
        eos_id = 3

        payload_len = self.max_chunk_len - 2
        step = payload_len - self.overlap

        chunks = []

        for start in range(0, len(token_ids), step):
            content = token_ids[start:start + payload_len]

            if not content:
                break

            chunks.append([bos_id] + content + [eos_id])

            if start + payload_len >= len(token_ids):
                break

            if len(chunks) >= self.max_chunks:
                break

        if not chunks:
            chunks = [[bos_id, eos_id]]

        # Multi-hot target vector: one position per statute.
        targets = torch.zeros(self.num_labels, dtype=torch.float32)

        if self.has_labels:
            for label in item["labels"]:
                label = int(label)

                if not 0 <= label < self.num_labels:
                    raise ValueError(
                        f"Label {label} outside range "
                        f"0..{self.num_labels - 1}, document={item['id']}"
                    )

                targets[label] = 1.0

        return {
            "id": item["id"],
            "chunks": chunks,
            "labels": targets,
        }


def collate_lsi(batch):
    batch_size = len(batch)
    max_chunks = max(len(item["chunks"]) for item in batch)

    max_len = max(
        len(chunk)
        for item in batch
        for chunk in item["chunks"]
    )

    pad_id = 0

    input_ids = torch.full(
        (batch_size, max_chunks, max_len),
        pad_id,
        dtype=torch.long,
    )

    attention_mask = torch.zeros_like(input_ids)

    chunk_mask = torch.zeros(
        batch_size, max_chunks, dtype=torch.bool
    )

    for i, item in enumerate(batch):
        for j, chunk in enumerate(item["chunks"]):
            n = len(chunk)

            input_ids[i, j, :n] = torch.tensor(
                chunk, dtype=torch.long
            )
            attention_mask[i, j, :n] = 1
            chunk_mask[i, j] = True

    labels = torch.stack([item["labels"] for item in batch])

    return {
        "ids": [item["id"] for item in batch],
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "chunk_mask": chunk_mask,
        "labels": labels,
    }

In [12]:
import torch
import torch.nn as nn

In [13]:
import json
from pathlib import Path

# Original dataset paths
train_path = train_file_paths["Phase_5_LSI/Train/train.json"]
val_path = val_file_paths["Phase_5_LSI/Train/dev.json"]

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def remove_duplicates(data, text_col="text"):
    seen = set()
    cleaned = []

    for item in data:
        text = item[text_col]  # Exact text; no normalization
        if text not in seen:
            seen.add(text)
            cleaned.append(item)

    return cleaned

# Load original data
train_data = load_json(train_path)
val_data = load_json(val_path)

print("Original train:", len(train_data))
print("Original validation:", len(val_data))

# Remove duplicates within each split
train_data = remove_duplicates(train_data)
val_data = remove_duplicates(val_data)

# Remove validation records that appear in train
train_texts = {item["text"] for item in train_data}

before = len(val_data)
val_data = [
    item for item in val_data
    if item["text"] not in train_texts
]

print("Cross-split duplicates removed from validation:", before - len(val_data))
print("Final train:", len(train_data))
print("Final validation:", len(val_data))

# Save cleaned JSON files
output_dir = Path("/kaggle/working/lsi_deduplicated")
output_dir.mkdir(parents=True, exist_ok=True)

clean_train_path = output_dir / "train_deduplicated.json"
clean_val_path = output_dir / "dev_deduplicated.json"

with open(clean_train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False)

with open(clean_val_path, "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False)

print("\nSaved files:")
print(clean_train_path)
print(clean_val_path)

Original train: 42750
Original validation: 10181
Cross-split duplicates removed from validation: 81
Final train: 42315
Final validation: 10037

Saved files:
/kaggle/working/lsi_deduplicated/train_deduplicated.json
/kaggle/working/lsi_deduplicated/dev_deduplicated.json


In [14]:
import torch
from torch.utils.data import DataLoader

train_dataset = LSIDataset(
    "/kaggle/working/lsi_deduplicated/train_deduplicated.json",
    converter=converter,
    has_labels=True,
)

dev_dataset = LSIDataset(
     "/kaggle/working/lsi_deduplicated/dev_deduplicated.json",
     converter=converter,
    has_labels=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_lsi,
)

val_loader = DataLoader(
    dev_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_lsi,
)

In [15]:

"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies =torch.tensor([1/base ** (2*i/d_head) for i in range(d_head//2)],dtype = torch.float32)
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    # TODO
    d_head = x.shape[-1]
    x1, x2 = x[...,:d_head//2], x[...,d_head//2:]
    return torch.cat([-x2, x1], dim=-1)



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rotated = x*cos + rotate_half(x)*sin
    return x_rotated



In [16]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F



class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None, is_causal=False):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        self.is_causal = is_causal
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)


        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, padding_mask = None, past_key_value=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        k = self.k_proj(x).view(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)

        v = self.v_proj(x).view(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)

        # Apply RoPE using absolute positions, before caching keys.
        if self.use_rope:
            cos = self.rope_cos[position_offset:position_offset + T]
            sin = self.rope_sin[position_offset:position_offset + T]
            q = apply_rope(
                q, cos, sin
            )
            k = apply_rope(
                k, cos, sin
            )

        # Append the new K/V to the cached prefix.
        if past_key_value is not None:
            past_k, past_v = past_key_value
            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        present = (k, v) if use_cache else None
        key_len = k.size(2)

        # Expand grouped KV heads for attention.
        if self.n_rep > 1:
            k_attn = k.repeat_interleave(self.n_rep, dim=1)
            v_attn = v.repeat_interleave(self.n_rep, dim=1)
        else:
            k_attn, v_attn = k, v

        scores = (q @ k_attn.transpose(-2, -1)) * self.attention_scale

        if self.is_causal:
            # During cached decoding, query positions start at position_offset.
            q_positions = torch.arange(
                position_offset, position_offset + T, device=x.device
            )
            k_positions = torch.arange(key_len, device=x.device)
            causal = k_positions[None, :] <= q_positions[:, None]
            scores = scores.masked_fill(
                ~causal[None, None, :, :], float("-inf")
            )

        if padding_mask is not None:
            if padding_mask.shape != (B, key_len):
                raise ValueError(
                    f"Expected padding_mask {(B, key_len)}, "
                    f"got {tuple(padding_mask.shape)}"
                )
        
            # True = padding; False = valid token.
            padding_mask = padding_mask.to(device=x.device, dtype=torch.bool)
            valid_keys = ~padding_mask
        
            if not valid_keys.any(dim=-1).all():
                raise ValueError("sequence does not have valid key")
        
            scores = scores.masked_fill(
                padding_mask[:, None, None, :], float("-inf")
            )

        weights = torch.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v_attn
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        if use_cache:
            return out, present
        return out


In [17]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F



class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_inputs, source_padding_mask=None, past_key_value=None, use_cache=False):
        B, T, C = x.shape
        B_enc, S, _ = encoder_inputs.shape
        if B != B_enc:
            raise ValueError("Encoder and decoder batch sizes must match.")

        q = self.q_proj(x).view(
            B, T, self.n_heads, self.d_head
        ).transpose(1, 2)

        # Reuse encoder projections after the first decoding step.
        if past_key_value is None:
            k = self.k_proj(encoder_inputs).view(
                B, S, self.n_kv_heads, self.d_head
            ).transpose(1, 2)
            v = self.v_proj(encoder_inputs).view(
                B, S, self.n_kv_heads, self.d_head
            ).transpose(1, 2)
        else:
            k, v = past_key_value

        present = (k, v) if use_cache else None

        if self.n_rep > 1:
            k_attn = k.repeat_interleave(self.n_rep, dim=1)
            v_attn = v.repeat_interleave(self.n_rep, dim=1)
        else:
            k_attn, v_attn = k, v

        scores = (q @ k_attn.transpose(-2, -1)) * self.attention_scale
        print("source_padding_mask shape:", source_padding_mask.shape)
        print("source_padding_mask sample:", source_padding_mask[:4])
        print("True counts per sequence:", source_padding_mask.sum(dim=-1)[:10])
        if source_padding_mask is not None:
            if source_padding_mask.shape != (B, S):
                raise ValueError("source_padding_mask must have shape (B, S).")
            source_padding_mask = source_padding_mask.to(
                device=x.device, dtype=torch.bool
            )
            if not source_padding_mask.any(dim=-1).all():
                raise ValueError("sequence does not have valid tokens")
            scores = scores.masked_fill(
                ~source_padding_mask[:, None, None, :], float("-inf")
            )

        weights = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = weights @ v_attn
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.resid_dropout(self.out_proj(out))

        if use_cache:
            return out, present
        return out



In [18]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook



def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE



class EncoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = EncoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, padding_mask=None):
        x = x + self.attn(self.ln1(x),padding_mask=padding_mask)
        x = x + self.ffn(self.ln2(x))
        return x   # unchanged from Question 1


class EncoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        
        

    def forward(self, idx: torch.Tensor, padding_mask=None) -> torch.Tensor:
        B, T = idx.shape
        
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError("padding_mask must have shape (B, T)")
            padding_mask = padding_mask.to(
                device=idx.device, dtype=torch.bool
            )
        x =self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
          x = x + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
          x=block(x, padding_mask=padding_mask)
        x = self.ln_f(x)
        return x # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")



In [19]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F





class DecoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.self_attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale, is_causal=True )
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn =  CrossAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = DecoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    # def forward(self, x, encoder_inputs,  target_padding_mask=None, source_padding_mask=None):
    #     x = x + self.self_attn(self.ln1(x), padding_mask=target_padding_mask)
    #     x = x + self.cross_attn( self.ln2(x), encoder_inputs, source_padding_mask=source_padding_mask)
    #     x = x + self.ffn(self.ln3(x))
    #     return x   # unchanged from Question 1
    def forward(self, x, encoder_inputs,  target_padding_mask=None, source_padding_mask=None, past_key_value=None, use_cache=False, position_offset=0):
        if past_key_value is None:
            past_self = past_cross = None
        else:
            past_self, past_cross = past_key_value
    
        if use_cache:
            attn, present_self = self.self_attn(
                self.ln1(x),
                padding_mask=target_padding_mask,
                past_key_value=past_self,
                use_cache=True,
                position_offset=position_offset,
            )
        else:
            attn = self.self_attn(
                self.ln1(x), padding_mask=target_padding_mask
            )
            present_self = None

        x = x + attn
    
        if use_cache:
            cross, present_cross = self.cross_attn(
                self.ln2(x),
                encoder_inputs,
                source_padding_mask=source_padding_mask,
                past_key_value=past_cross,
                use_cache=True,
            )
        else:
            cross = self.cross_attn(
                self.ln2(x), encoder_inputs,
                source_padding_mask=source_padding_mask,
            )
            present_cross = None
    
        x = x + cross
        x = x + self.ffn(self.ln3(x))
    
        if use_cache:
            return x, (present_self, present_cross)
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor, encoder_inputs:torch.Tensor, target_padding_mask=None, source_padding_mask=None,past_key_values=None, use_cache=False) -> torch.Tensor:
        B, T = idx.shape

        if past_key_values is not None and len(past_key_values) != len(self.blocks):
            raise ValueError("Cache must contain one entry per decoder layer.")
        past_len = (
            0 if past_key_values is None
            else past_key_values[0][0][0].size(2)
        )
        total_len = past_len + T
        if total_len > self.cfg.max_seq_len:
            raise ValueError(
                f"Decoder length {total_len} exceeds "
                f"max_seq_len={self.cfg.max_seq_len}. "
                "Reset or trim the KV cache."
            )
        # Validate the complete decoder key mask.
        if target_padding_mask is not None:
            if target_padding_mask.shape != (B, total_len):
                raise ValueError(
                    f"Expected target mask {(B, total_len)}, "
                    f"got {tuple(target_padding_mask.shape)}"
                )
    
            target_padding_mask = target_padding_mask.to(
                device=idx.device,
                dtype=torch.bool,
            )
            if not target_padding_mask.any(dim=-1).all():
                raise ValueError(
                    "Every decoder sequence needs at least one valid token."
                )

        x = self.tok_emb(idx)
    
        if self.cfg.pos_emb_type == "sinusoidal":
            end_pos = past_len + T
            if end_pos > self.pos_emb.size(0):
                raise ValueError("Increase cfg.max_seq_len for this sequence.")
            x = x + self.pos_emb[past_len:end_pos].unsqueeze(0)
    
        presents = []
    
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
    
            if use_cache:
                x, present = block(
                    x, encoder_inputs,
                    target_padding_mask=target_padding_mask,
                    source_padding_mask=source_padding_mask,
                    past_key_value=past,
                    use_cache=True,
                    position_offset=past_len,
                )
                presents.append(present)
            else:
                x = block(
                    x, encoder_inputs,
                    target_padding_mask=target_padding_mask,
                    source_padding_mask=source_padding_mask,
                )
    
        logits = self.head(self.ln_f(x))
    
        if use_cache:
            return logits, tuple(presents)
        return logits
    
            # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")
    


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class EncoderDecoderTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg

        self.encoder = EncoderLM(cfg)
        self.decoder = DecoderLM(cfg)

    def forward(self, src, tgt, source_padding_mask=None, target_padding_mask=None,past_key_values=None,use_cache=False):
        """
        src: (B, S) source token IDs
        tgt: (B, T) decoder input token IDs

        source_padding_mask: (B, S), True = valid token
        target_padding_mask: (B, T), True = valid token

        Returns:
            logits: (B, T, vocab_size)
        """

        # 1. Encode source tokens
        encoder_outputs = self.encoder(
                            src,
                            padding_mask=source_padding_mask,
                        )

        # 2. Decode target tokens using encoder outputs
        logits = self.decoder(
                    tgt,
                    encoder_outputs,
                    target_padding_mask=target_padding_mask,
                    source_padding_mask=source_padding_mask,
                )
        if use_cache:
           logits, present_key_values = result
           return logits, present_key_values

        return logits

In [21]:
class LoraLinear(nn.Module):
    def __init__(self, layer, rank=8, alpha=16, dropout=0.05):
        super().__init__()

        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout)

        # Freeze original layer
        for p in self.layer.parameters():
            p.requires_grad = False

        self.in_features = layer.in_features
        self.out_features = layer.out_features

        # A: [rank, in_features]
        self.A = nn.Parameter(
            torch.empty(rank, self.in_features)
        )

        # B: [out_features, rank]
        self.B = nn.Parameter(
            torch.zeros(self.out_features, rank)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=5**0.5
        )

    def forward(self, x):

        # Original path: W0 x
        W0x = self.layer(x)

        # LoRA path: BAx
        Ax = F.linear(self.dropout(x), self.A)
        BAx = F.linear( Ax, self.B)

        return W0x + (self.alpha / self.rank) * BAx

In [22]:

class LoraModel(nn.Module):
    def __init__(
        self,
        base_model,
        d_model,
        num_classes,
        rank=8,
        alpha=16,
        lora_dropout=0.05,
        dropout=0.2,
        use_pos=False,
        projections=("q_proj", "k_proj"),
    ):
        super().__init__()

        self.model = base_model
        self.use_pos = use_pos
        self.drop = nn.Dropout(dropout)

        # Freeze original Phase-1 parameters.
        for param in self.model.parameters():
            param.requires_grad = False

        # Insert trainable LoRA adapters.
        self._add_lora(
            rank=rank,
            alpha=alpha,
            dropout=lora_dropout,
            projections=projections,
        )

        # Same classification head as RRClassifier.
        self.head = nn.Linear(
            d_model + int(use_pos),
            num_classes,
        )

    def _add_lora(self, rank, alpha, dropout, projections):
        for module in self.model.modules():
            for proj_name in projections:

                # Feed-forward layers, if requested.
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2

                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LoraLinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            )

                # Attention projections.
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)

                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LoraLinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                            ),
                        )
    # def get_hidden_states(self, idx, padding_mask=None):
    #     B, T = idx.shape
    #     x = self.model.tok_emb(idx)
    #     if self.model.cfg.pos_emb_type == "sinusoidal":
    #         x = x + self.model.pos_emb[:T]
    #     if padding_mask is not None:
    #         if padding_mask.shape != (B, T):
    #             raise ValueError("padding_mask must have shape (B, T)")
    #         padding_mask = padding_mask.to(device=idx.device, dtype=torch.bool)
    #     for block in self.model.blocks:
    #         x, _ = block(x, padding_mask=padding_mask)
    #     return self.model.ln_f(x)
        # return x  # Shape: [B, T, d_model]
    def encode(self, input_ids, padding_mask):
        hidden = self.model(input_ids, padding_mask)
    
        if hidden is None:
            raise RuntimeError(
                "get_hidden_states() returned None. "
                "Check its return statement."
            )
    
        valid = (~padding_mask.bool()).unsqueeze(-1).to(hidden.dtype)
        counts = valid.sum(dim=1)
    
        pooled = (hidden * valid).sum(dim=1) / counts.clamp(min=1)
    
        return pooled


    def forward(self, input_ids, padding_mask):
        pooled = self.encode(
            input_ids=input_ids,
            padding_mask=padding_mask,
        )
    
        return self.head(self.drop(pooled))
                    
        


In [23]:
class LormaLinear(nn.Module):
    def __init__(self, layer, rank=16, alpha=32, dropout=0.05, variant = "plus"):
        super().__init__()
        if variant not in {"plus", "pi"}:
            raise ValueError("Only phi or plus supported")
        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        self.variant = variant
        self.dropout = nn.Dropout(dropout)

        for p in self.layer.parameters():
            p.requires_grad = False
        d = layer.out_features
        self.A = nn.Parameter(torch.empty(rank, d))
        self.B = nn.Parameter(torch.empty(d, rank))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=5**0.5)
            nn.init.zeros_(self.B)
        else:
            nn.init.normal_(self.B)
            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A.zero_()
                self.A[0, 0] = 1.0
    @staticmethod
    def inflate_pi(BA):
        """Cyclically rotate row i by i positions."""
        d = BA.shape[0]
        return torch.stack(
            [torch.roll(BA[i], shifts=i, dims=0) for i in range(d)],
            dim=0,
        )
    def forward(self, x):
        base = self.layer(x)

        # Apply BA to the pretrained output
        z = self.dropout(base)
        if self.variant == "plus":
            Az = F.linear(z, self.A)
            BAz = F.linear(Az, self.B)
    
            # h = (I + alpha/rank * BA) W0 x
            output = base + (self.alpha / self.rank) * BAz
            
    
            return output
        else:
            BA = self.B @ self.A

            # Permutation-based rank inflation
            BA_pi = self.inflate_pi(BA)
    
            # h = (I_pi(BA) @ W0) x
            # Equivalent to applying BA_pi to W0 x
            adapted = F.linear(z, BA_pi)
    
            return base + (self.alpha / self.rank) * adapted

In [24]:
class LormaModel(nn.Module):
    def __init__(self, model, d_model, rank = 8, alpha =16, dropout =0.05,lorma_dropout =0.05, labels =25, variant="plus",use_pos =False, projections=("q_proj", "k_proj")):
        super().__init__()
        self.model = model
        self.d_model = d_model
        self.projections = tuple(projections)
        self.drop = nn.Dropout(dropout)
        self.variant = variant
        self.use_pos =use_pos
        # Freeze Phase-1 model
        for param in self.model.parameters():
            param.requires_grad = False
        self._add_lorma(rank, alpha, lorma_dropout)
        self.ner_head = nn.Linear(d_model, labels)


    def _add_lorma(self, rank, alpha, dropout):
        for module in list(self.model.modules()):
            for proj_name in self.projections:
    
                if proj_name in ("net0", "net1"):
                    if hasattr(module, "net"):
                        idx = 0 if proj_name == "net0" else 2
    
                        if isinstance(module.net[idx], nn.Linear):
                            module.net[idx] = LormaLinear(
                                module.net[idx],
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                                variant = self.variant
                            )
    
                elif hasattr(module, proj_name):
                    layer = getattr(module, proj_name)
    
                    if isinstance(layer, nn.Linear):
                        setattr(
                            module,
                            proj_name,
                            LormaLinear(
                                layer,
                                rank=rank,
                                alpha=alpha,
                                dropout=dropout,
                                variant = self.variant
                            ),
                        )
    def get_hidden_states(self, idx, padding_mask=None):
        B, T = idx.shape
        x = self.model.tok_emb(idx)
        if self.model.cfg.pos_emb_type == "sinusoidal":
            x = x + self.model.pos_emb[:T]
        if padding_mask is not None:
            if padding_mask.shape != (B, T):
                raise ValueError("padding_mask must have shape (B, T)")
            padding_mask = padding_mask.to(device=idx.device, dtype=torch.bool)
        for block in self.model.blocks:
            x, _ = block(x, padding_mask=padding_mask)
        return self.model.ln_f(x)

    def encode(self, input_ids, padding_mask):
        hidden = self.get_hidden_states(input_ids, padding_mask)
    
        if hidden is None:
            raise RuntimeError(
                "get_hidden_states() returned None. "
                "Check its return statement."
            )
    
        valid = (~padding_mask.bool()).unsqueeze(-1).to(hidden.dtype)
        counts = valid.sum(dim=1)
    
        pooled = (hidden * valid).sum(dim=1) / counts.clamp(min=1)
    
        return pooled


    def forward(self, input_ids, padding_mask):
        pooled = self.encode(
            input_ids=input_ids,
            padding_mask=padding_mask,
        )
    
        return self.head(self.drop(pooled))
                    


In [25]:

import torch
import torch.nn as nn


class HierarchicalLoraLSIAttention(nn.Module):
    def __init__(
        self,
        lora_classifier,
        num_labels=100,
        gru_hidden=256,
        attention_dim=256,
        dropout=0.2,
        freeze_lora_classifier=True,
    ):
        super().__init__()

        self.chunk_encoder = lora_classifier
        self.freeze_lora_classifier = freeze_lora_classifier
        self.num_labels = num_labels

        d_model = self.chunk_encoder.model.cfg.d_model

        self.bigru = nn.GRU(
            input_size=d_model,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
        )

        # Additive attention over BiGRU chunk representations
        self.attention = nn.Sequential(
            nn.Linear(2 * gru_hidden, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1, bias=False),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(2 * gru_hidden, gru_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden, num_labels),
        )

        if freeze_lora_classifier:
            for p in self.chunk_encoder.parameters():
                p.requires_grad = False

            # Re-enable LoRA adapter parameters
            for module in self.chunk_encoder.modules():
                if isinstance(module, LoraLinear):
                    for p in module.parameters():
                        p.requires_grad = True

    def train(self, mode=True):
        super().train(mode)

        # Keep the encoder in eval mode when its base is frozen.
        if self.freeze_lora_classifier:
            self.chunk_encoder.eval()

        return self

    def forward(self, input_ids, attention_mask, chunk_mask):
        # input_ids, attention_mask: [B, K, T]
        # chunk_mask: [B, K]
        B, K, T = input_ids.shape

        flat_ids = input_ids.reshape(B * K, T)
        flat_attention = attention_mask.reshape(B * K, T).bool()
        valid_chunks = chunk_mask.reshape(B * K).bool()

        chunk_dim = self.chunk_encoder.model.cfg.d_model

        chunk_vectors = (
            self.chunk_encoder.model.tok_emb.weight.new_zeros(
                B * K, chunk_dim
            )
        )

        if valid_chunks.any():
            ids = flat_ids[valid_chunks]
            padding_mask = ~flat_attention[valid_chunks]

            vectors = self.chunk_encoder.encode(
                input_ids=ids,
                padding_mask=padding_mask,
            )

            chunk_vectors[valid_chunks] = vectors.to(
                chunk_vectors.dtype
            )

        chunk_vectors = chunk_vectors.reshape(B, K, chunk_dim)

        lengths = chunk_mask.sum(dim=1).long()

        if (lengths <= 0).any():
            raise ValueError(
                "Each document must contain at least one chunk."
            )

        # Pack valid chunk sequences for the BiGRU
        packed = nn.utils.rnn.pack_padded_sequence(
            chunk_vectors,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        # Unpack: [B, K, 2 * gru_hidden]
        packed_outputs, _ = self.bigru(packed)

        gru_outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=K,
        )

        # Attention score for each chunk: [B, K]
        scores = self.attention(gru_outputs).squeeze(-1)

        # Mask padded chunks so they receive zero attention
        scores = scores.masked_fill(
            ~chunk_mask.bool(), float("-inf")
        )

        # Attention weights sum to 1 over valid chunks
        attention_weights = torch.softmax(scores, dim=1)

        # Weighted document representation: [B, 2 * gru_hidden]
        document_vector = torch.sum(
            gru_outputs * attention_weights.unsqueeze(-1),
            dim=1,
        )

        # Raw logits: [B, num_labels]
        logits = self.classifier(document_vector)

        return logits


In [26]:
cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=6,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)
model_phase1 = EncoderDecoderTransformer(cfg)

model_phase1 = model_phase1.to(device)

In [27]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())
print("total parameters : ",count_parameters(model_phase1))

total parameters :  77486080


In [28]:
checkpoint = torch.load(
    "/kaggle/input/datasets/sai123451/encoder-decoder-model/encoder_decoder_final.pt",
    map_location=device,
    weights_only=False
)




state_dict = checkpoint["model_state_dict"]

new_state_dict = {}

for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    new_state_dict[key] = value

model_phase1.load_state_dict(new_state_dict)
encoder_model = model_phase1.encoder

In [29]:
batch = next(iter(train_loader))

print("Input IDs:", batch["input_ids"].shape)
print("Attention mask:", batch["attention_mask"].shape)
print("Chunk mask:", batch["chunk_mask"].shape)
print("Labels:", batch["labels"].shape)
print("Document IDs:", batch["ids"])

Input IDs: torch.Size([2, 3, 1024])
Attention mask: torch.Size([2, 3, 1024])
Chunk mask: torch.Size([2, 3])
Labels: torch.Size([2, 100])
Document IDs: ['141022057', '1602048']


In [30]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model, d_model, rank = 8, alpha =16, dropout =0.05, labels =25, variant="plus", projections=("q_proj", "k_proj")
lora_classifier = LoraModel(
    base_model=encoder_model,
    d_model=model_phase1.cfg.d_model,
    num_classes=2,
    rank=16,
    alpha=32,
    lora_dropout=0.05,
    dropout=0.2,
    use_pos=False,
    projections=("q_proj", "k_proj","v_proj","o_proj"),
)

model = HierarchicalLoraLSIAttention(
    lora_classifier=lora_classifier,
    gru_hidden=256,
    dropout=0.2,
    freeze_lora_classifier=False,  # Enable LoRA adapter training
).to(device)

# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

# print("Device:", device)
# print("Number of GPUs:", torch.cuda.device_count())

In [31]:
# if torch.cuda.device_count() > 1:
#     model = nn.DataParallel(model)

# print("GPUs available:", torch.cuda.device_count())
# print("Model device:", device)

In [32]:

import torch
from tqdm.auto import tqdm


def train_one_epoch(
    model, train_loader, optimizer, criterion, device, scaler
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    progress = tqdm(train_loader, desc="LSI Training", leave=False)

    for batch in progress:
        input_ids = batch["input_ids"].to(
            device, non_blocking=True
        )
        attention_mask = batch["attention_mask"].to(
            device, non_blocking=True
        )
        chunk_mask = batch["chunk_mask"].to(
            device, non_blocking=True
        )

        # Multi-label targets: [B, NUM_LABELS], float values 0/1
        labels = batch["labels"].to(
            device=device, dtype=torch.float32, non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                chunk_mask=chunk_mask,
            )

            if logits.shape != labels.shape:
                raise ValueError(
                    f"Logits shape {logits.shape} != "
                    f"labels shape {labels.shape}"
                )

            loss = criterion(logits.float(), labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        total_loss += loss.detach().item() * batch_size
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{total_loss / max(total_samples, 1):.4f}"
        )

    return {
        "loss": total_loss / max(total_samples, 1)
    }

import numpy as np
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


@torch.no_grad()
def evaluate(model, val_loader, criterion, device, threshold=0.5):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_predictions = []
    all_labels = []

    for batch in tqdm(val_loader, desc="LSI Validation", leave=False):
        input_ids = batch["input_ids"].to(
            device, non_blocking=True
        )
        attention_mask = batch["attention_mask"].to(
            device, non_blocking=True
        )
        chunk_mask = batch["chunk_mask"].to(
            device, non_blocking=True
        )
        labels = batch["labels"].to(
            device=device, dtype=torch.float32, non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                chunk_mask=chunk_mask,
            )
            loss = criterion(logits.float(), labels)

        # Independent probability for each statute
        probabilities = torch.sigmoid(logits.float())
        predictions = (probabilities >= threshold).to(torch.int32)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_predictions.append(predictions.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    y_pred = np.concatenate(all_predictions, axis=0)
    y_true = np.concatenate(all_labels, axis=0).astype(np.int32)

    return {
        "loss": total_loss / max(total_samples, 1),
        "micro_f1": f1_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "micro_precision": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "micro_recall": recall_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "predictions": y_pred,
        "labels": y_true,
    }


In [33]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Move model to the primary GPU
model = model.to(device)

# 2. Enable DataParallel when multiple GPUs are available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
else:
    print("Using a single GPU")

# 3. Define optimizer AFTER wrapping the model
lora_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue

    if name.endswith(".A") or name.endswith(".B"):
        lora_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": lora_params, "lr": 2e-4},
        {"params": head_params, "lr": 1e-3},
    ],
    weight_decay=0.01,
)

criterion = nn.BCEWithLogitsLoss()

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)

Using 2 GPUs


In [34]:
from pathlib import Path
import torch

# Directory to save model-only checkpoints
checkpoint_dir = Path("/kaggle/working/lsi_checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

NUM_EPOCHS = 5

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n{'=' * 20} Epoch {epoch}/{NUM_EPOCHS} {'=' * 20}")

    # Train
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler
    )

    # Validate
    val_metrics = evaluate(
        model, val_loader, criterion, device, threshold=0.5
    )

    # Print metrics
    print(f"Train loss:        {train_metrics['loss']:.6f}")
    print(f"Validation loss:   {val_metrics['loss']:.6f}")
    print(f"Micro-F1:          {val_metrics['micro_f1']:.6f}")
    print(f"Macro-F1:          {val_metrics['macro_f1']:.6f}")
    print(f"Micro precision:   {val_metrics['micro_precision']:.6f}")
    print(f"Micro recall:      {val_metrics['micro_recall']:.6f}")

    # Save ONLY model weights for this epoch
    model_to_save = model.module if isinstance(
        model, torch.nn.DataParallel
    ) else model

    checkpoint_path = checkpoint_dir / f"lsiattention_epoch_{epoch}.pt"
    torch.save(model_to_save.state_dict(), checkpoint_path)

    print(f"Model checkpoint saved: {checkpoint_path}")


==================== Epoch 1/5 ====================


LSI Training:   0%|          | 0/21158 [00:00<?, ?it/s]

LSI Validation:   0%|          | 0/5019 [00:00<?, ?it/s]

Train loss:        0.121940
Validation loss:   0.117568
Micro-F1:          0.158106
Macro-F1:          0.047333
Micro precision:   0.691200
Micro recall:      0.089262
Model checkpoint saved: /kaggle/working/lsi_checkpoints/lsiattention_epoch_1.pt

==================== Epoch 2/5 ====================


LSI Training:   0%|          | 0/21158 [00:00<?, ?it/s]

LSI Validation:   0%|          | 0/5019 [00:00<?, ?it/s]

... (6 outputs left out)

Train loss:        0.105060
Validation loss:   0.109873
Micro-F1:          0.292701
Macro-F1:          0.119275
Micro precision:   0.683566
Micro recall:      0.186220
Model checkpoint saved: /kaggle/working/lsi_checkpoints/lsiattention_epoch_4.pt

==================== Epoch 5/5 ====================


LSI Training:   0%|          | 0/21158 [00:00<?, ?it/s]

LSI Validation:   0%|          | 0/5019 [00:00<?, ?it/s]

Train loss:        0.103543
Validation loss:   0.109093
Micro-F1:          0.284692
Macro-F1:          0.120888
Micro precision:   0.703607
Micro recall:      0.178448
Model checkpoint saved: /kaggle/working/lsi_checkpoints/lsiattention_epoch_5.pt


In [35]:
import torch
import numpy as np

if isinstance(model, torch.nn.DataParallel):
    model = model.module

model.eval()

all_y_probs = []
all_y_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        chunk_mask = batch["chunk_mask"].to(device, non_blocking=True)

        labels = batch["labels"].to(
            device=device,
            dtype=torch.float32,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                chunk_mask=chunk_mask,
            )

        # Convert logits to multi-label probabilities
        probs = torch.sigmoid(logits.float())

        all_y_probs.append(probs.cpu())
        all_y_labels.append(labels.cpu())

# Combine all validation batches
y_probs = torch.cat(all_y_probs, dim=0).numpy()
y_labels = torch.cat(all_y_labels, dim=0).numpy()

print("y_probs shape:", y_probs.shape)
print("y_labels shape:", y_labels.shape)

y_probs shape: (10037, 100)
y_labels shape: (10037, 100)


In [36]:
from sklearn.metrics import f1_score

for threshold in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    y_pred = (y_probs >= threshold).astype(np.int32)

    micro_f1 = f1_score(
        y_labels, y_pred, average="micro", zero_division=0
    )
    macro_f1 = f1_score(
        y_labels, y_pred, average="macro", zero_division=0
    )

    print(
        f"Threshold={threshold:.2f} | "
        f"Micro-F1={micro_f1:.4f} | "
        f"Macro-F1={macro_f1:.4f}"
    )

Threshold=0.05 | Micro-F1=0.3189 | Macro-F1=0.2341
Threshold=0.10 | Micro-F1=0.3965 | Macro-F1=0.2461
Threshold=0.15 | Micro-F1=0.4285 | Macro-F1=0.2328
Threshold=0.20 | Micro-F1=0.4393 | Macro-F1=0.2179
Threshold=0.25 | Micro-F1=0.4343 | Macro-F1=0.1999
Threshold=0.30 | Micro-F1=0.4162 | Macro-F1=0.1817
Threshold=0.40 | Micro-F1=0.3574 | Macro-F1=0.1526
Threshold=0.50 | Micro-F1=0.2847 | Macro-F1=0.1209


In [ ]:
test_dataset = LSIDataset(
     test_file_paths["Phase_5_LSI/Test/test.json"],
    converter=converter,
    has_labels=False, 
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_lsi,
)

In [ ]:

model.eval()
all_test_probs = []
all_test_ids = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        chunk_mask = batch["chunk_mask"].to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda"),
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                chunk_mask=chunk_mask,
            )

        probs = torch.sigmoid(logits.float())
        all_test_probs.append(probs.cpu().numpy())
        all_test_ids.extend(batch["ids"])

test_probs = np.concatenate(all_test_probs, axis=0)
test_preds = (test_probs >= 0.2).astype(np.int32)

assert len(all_test_ids) == len(test_preds)

submission = [
    {
        "id": case_id,
        "pred": np.where(pred == 1)[0].tolist(),
    }
    for case_id, pred in zip(all_test_ids, test_preds)
]

with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=2)

print(json.dumps(submission[:2], indent=2))
print("Total cases:", len(submission))